# CS 577 Homework 1: Tools and Tensor Fluency

**Student:** Irma Modzgvrishvili
**Hawk username:** imodzgvrishvili@hawk.illinoistech.edu

Complete this notebook together with `hw1_utils.py`. Restart the kernel and run all cells before submission. Keep all outputs visible. Written derivations and the final results table belong in the LaTeX PDF.

## Part 0 - Environment and reproducibility


In [1]:
import os
import platform
import sys
import timeit

import numpy as np
import torch

import hw1_utils as h

print("Python:", sys.version)
print("OS:", platform.platform())
print("NumPy:", np.__version__)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Selected device:", DEVICE)
print("NumPy example dtype:", np.zeros(1).dtype)
print("PyTorch default dtype:", torch.get_default_dtype())


Python: 3.13.5 (main, Jun 11 2025, 15:36:57) [Clang 17.0.0 (clang-1700.0.13.3)]
OS: macOS-15.7.4-arm64-arm-64bit-Mach-O
NumPy: 2.5.2
PyTorch: 2.13.0
CUDA available: False
Selected device: cpu
NumPy example dtype: float64
PyTorch default dtype: torch.float32


In [2]:
# Demonstrate reproducibility after implementing set_all_seeds.
h.set_all_seeds(577)
numpy_draw_1 = np.random.randn(4)
torch_draw_1 = torch.randn(4)

h.set_all_seeds(577)
numpy_draw_2 = np.random.randn(4)
torch_draw_2 = torch.randn(4)

print(numpy_draw_1)
print(numpy_draw_2)
print(torch_draw_1)
print(torch_draw_2)
assert np.array_equal(numpy_draw_1, numpy_draw_2)
assert torch.equal(torch_draw_1, torch_draw_2)


[ 0.41859251  0.20710891  2.06338464 -1.46638813]
[ 0.41859251  0.20710891  2.06338464 -1.46638813]
tensor([ 0.4522,  0.0206, -0.9701, -0.3854])
tensor([ 0.4522,  0.0206, -0.9701, -0.3854])


### Reproducibility note

Setting a `seed` helps make the random parts of training `start` from the same point, which makes experiments `easier` to compare. in other words: Setting a seed helps reproduce the same randomly generated values and random choices across repeated runs of the experiment. This helps us understand whether the result changed because of changes in the `model` or because of `randomness`. However, a `seed` does not guarantee exactly the same result on `every machine`, because `different devices`, `library version`s, and `parallel operations` may perform calculations `differently`. Since floating-point calculations have limited precision, these small differences can produce slightly different results.


## Part 2 - NumPy tensors and vectorization


In [3]:
X = np.array([
    [ 1.0,  2.0, -1.0],
    [ 0.0,  1.0,  3.0],
    [ 2.0, -2.0,  1.0],
    [-1.0,  0.0,  2.0],
])
W = np.array([[0.5, -1.0], [1.5, 0.0], [-0.5, 2.0]])
b = np.array([0.25, -0.75])

Y = h.affine_numpy(X, W, b)
print("shapes:", X.shape, W.shape, b.shape, Y.shape)
print(Y)


shapes: (4, 3) (3, 2) (2,) (4, 2)
[[ 4.25 -3.75]
 [ 0.25  5.25]
 [-2.25 -0.75]
 [-1.25  4.25]]


`X @ W` is matrix multiplication. Since `X`has shape (4,3) and `W` has shape (3,2), the inner dimensions match (3 = 3), so the result has shape (4,2). The bias `b` has shape (2,). Broadcasting compares dimensions from right to left, so (2,) is treated like (1,2). Since 2 = 2 and 1 can expand to 4, `b` is broadcast to shape (4,2) and added to every row of `X @ W`.

In [4]:
Z, mean, std = h.standardize_columns(X)
print("mean:", mean)
print("std:", std)
print("Z:\n", Z)
print("check mean:", Z.mean(axis=0))
print("check std:", Z.std(axis=0, ddof=0))


mean: [0.5  0.25 1.25]
std: [1.11803399 1.47901995 1.47901995]
Z:
 [[ 0.4472136   1.18321596 -1.52127766]
 [-0.4472136   0.50709255  1.18321596]
 [ 1.34164079 -1.52127766 -0.16903085]
 [-1.34164079 -0.16903085  0.50709255]]
check mean: [ 0.00000000e+00 -1.38777878e-17  0.00000000e+00]
check std: [1. 1. 1.]


In [5]:
A = np.array([[0., 0.], [1., 0.], [0., 2.]])
B = np.array([[1., 1.], [-1., 0.]])
D2 = h.pairwise_squared_distances(A, B)
print("expanded shapes:", A[:, None, :].shape, B[None, :, :].shape)
print(D2)


expanded shapes: (3, 1, 2) (1, 2, 2)
[[2. 1.]
 [1. 4.]
 [2. 5.]]


In [6]:
def pairwise_loop(A, B):
    out = np.empty((A.shape[0], B.shape[0]))
    for i in range(A.shape[0]):
        for j in range(B.shape[0]):
            out[i, j] = np.sum((A[i] - B[j]) ** 2)
    return out

rng = np.random.default_rng(577)
A_bench = rng.normal(size=(200, 16))
B_bench = rng.normal(size=(250, 16))
np.testing.assert_allclose(pairwise_loop(A_bench[:5], B_bench[:7]),
                           h.pairwise_squared_distances(A_bench[:5], B_bench[:7]))

loop_times = timeit.repeat(lambda: pairwise_loop(A_bench, B_bench), repeat=5, number=1)
vec_times = timeit.repeat(lambda: h.pairwise_squared_distances(A_bench, B_bench), repeat=5, number=1)
print("loop median:", np.median(loop_times))
print("vectorized median:", np.median(vec_times))
print("local ratio:", np.median(loop_times) / np.median(vec_times))


loop median: 0.17025283304974437
vectorized median: 0.005073208129033446
local ratio: 33.559205283813405


### NumPy interpretation

Record your broadcasting explanation and careful timing interpretation here before transferring the polished version to the PDF.


## Part 3 - PyTorch tensors and a minimal model


In [7]:
Xt = torch.tensor(X, dtype=torch.float64, device=DEVICE)
Wt = torch.tensor(W, dtype=torch.float64, device=DEVICE)
bt = torch.tensor(b, dtype=torch.float64, device=DEVICE)
Yt = Xt @ Wt + bt
max_difference = np.max(np.abs(Yt.detach().cpu().numpy() - Y))
print("max NumPy/PyTorch difference:", max_difference)


max NumPy/PyTorch difference: 0.0


In [8]:
model = h.LinearRegressor().to(DEVICE)
with torch.no_grad():
    model.linear.weight.zero_()
    model.linear.bias.zero_()

x_train = torch.linspace(-1, 1, 41, device=DEVICE).reshape(-1, 1)
y_train = 2.5 * x_train - 0.4
losses = h.train_tiny_regressor(model, x_train, y_train, steps=100, lr=0.1)

print("initial loss:", losses[0])
print("final loss:", losses[-1])
print("weight:", model.linear.weight.item())
print("bias:", model.linear.bias.item())
print("selected losses:", {i: losses[i] for i in [0, 9, 49, 99]})


NotImplementedError: 

### Dtype/device note

Replace this paragraph with your 2-4 sentence explanation.


## Part 4 - Three views of the same gradient


In [ ]:
Xg = np.array([[1.0, -1.0], [0.5, 2.0], [-2.0, 1.0]])
yg = np.array([1.0, 0.0, -1.0])
w0 = np.array([0.2, -0.3])
lam = 0.1

analytic = h.mse_l2_grad_numpy(Xg, yg, w0, lam)
objective = lambda w: h.mse_l2_objective_numpy(Xg, yg, w, lam)
finite_difference = h.finite_difference_gradient(objective, w0, epsilon=1e-5)

Xg_t = torch.tensor(Xg, dtype=torch.float64)
yg_t = torch.tensor(yg, dtype=torch.float64)
w_t = torch.tensor(w0, dtype=torch.float64, requires_grad=True)
residual_t = Xg_t @ w_t - yg_t
objective_t = torch.mean(residual_t ** 2) + 0.5 * lam * torch.sum(w_t ** 2)
objective_t.backward()
autograd = w_t.grad.detach().numpy()

print("objective:", objective(w0))
print("analytic:", analytic)
print("finite difference:", finite_difference)
print("autograd:", autograd)
print("analytic vs finite difference:", np.max(np.abs(analytic - finite_difference)))
print("analytic vs autograd:", np.max(np.abs(analytic - autograd)))


### Gradient comparison

Replace this paragraph with your explanation of agreement and finite-difference error.


## Part 5 - Reflection

1. **Shape/broadcasting mistake:** TODO
2. **Gradient validation method:** TODO
3. **Workflow change before HW2:** TODO


In [ ]:
# Run the public tests from within the notebook environment.
!python -m unittest -v test_hw1_public.py
